# Auto populate descriptions inside your semantic model

In [ ]:
## Install necessary packages
%pip install semantic-link-labs

In [ ]:
# Code to autopopulate
# All lines with  ** - replace the values

import pandas as pd
from sempy_labs.tom import connect_semantic_model

DATASET   = "cruz-semantic-model"   # as it appears in the workspace **
WORKSPACE = "smec-dataplatform-e5" **
DESC_FILE = "Files/landing-zone/2_Data Dictionary.csv"   # lakehouse path, or use abfss://... **

# --- load the external file (CSV here; use pd.read_excel for .xlsx) ---
meta = pd.read_csv(f"/lakehouse/default/{DESC_FILE}")
meta["column_name"] = meta["column_name"].fillna("").astype(str).str.strip()
meta["table_name"]  = meta["table_name"].astype(str).str.strip()
meta["description"] = meta["description"].fillna("").astype(str)

# index for quick lookup
tbl_desc = {r.table_name: r.description
            for r in meta.itertuples() if r.column_name == "" and r.description}
col_desc = {(r.table_name, r.column_name): r.description
            for r in meta.itertuples() if r.column_name and r.description}

applied, missing = 0, []

with connect_semantic_model(dataset=DATASET, workspace=WORKSPACE, readonly=False) as tom:
    for t in tom.model.Tables:
        # table-level description
        if t.Name in tbl_desc:
            t.Description = tbl_desc[t.Name]
            applied += 1
        # column-level descriptions
        for c in t.Columns:
            key = (t.Name, c.Name)
            if key in col_desc:
                c.Description = col_desc[key]
                applied += 1

    # report anything in the file that didn't match a real object
    model_tables = {t.Name for t in tom.model.Tables}
    model_cols   = {(t.Name, c.Name) for t in tom.model.Tables for c in t.Columns}
    missing  = [k for k in tbl_desc if k not in model_tables]
    missing += [k for k in col_desc if k not in model_cols]

print(f"Applied {applied} descriptions.")
if missing:
    print("No matching object for:", missing)
# Changes are committed automatically when the `with` block exits.